# Tanimoto Similarity Sanity Check

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

pd.set_option("display.max_colwidth", 120)

In [2]:
SPLIT_CSV_PATH = Path("processed/split_manifest.csv")
LIGAND_COLUMN = "ligand"
SPLIT_COLUMN = "split"

FP_RADIUS = 2
FP_SIZE = 248

DIVERSITY_SPLITS = ["train", "val", "test"]
DIVERSITY_THRESHOLD = 0.3
MAX_LIGANDS_PER_SPLIT_FOR_DIVERSITY = 500
DIVERSITY_RANDOM_SEED = 7
TOP_PAIRS_PER_SPLIT = 5

QUERY_LIGANDS = [
    "Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1",
    "C/N=C1/N/C(=C/c2c[nH]c3cc(Br)ccc23)C(=O)N1C",
    "C1=C(c2ccoc2)C2CCN1CC2",
]

TOP_K = 10

In [3]:
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE)


def try_canonicalize_smiles(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def fingerprint_from_canonical_smiles(smiles: str):
    if smiles is None:
        return None
    mol = Chem.MolFromSmiles(smiles)
    return fpgen.GetFingerprint(mol)


def load_split_df(path: Path, ligand_column: str = LIGAND_COLUMN, split_column: str = SPLIT_COLUMN) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Split CSV not found: {path.resolve()}")

    df = pd.read_csv(path)
    missing = [col for col in (ligand_column, split_column) if col not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. Available columns: {list(df.columns)}"
        )

    df = df.copy()
    df[ligand_column] = df[ligand_column].astype(str).str.strip()
    df["input_ligand"] = df[ligand_column]
    df["canonical_ligand"] = df["input_ligand"].map(try_canonicalize_smiles)
    df["is_valid_smiles"] = df["canonical_ligand"].notna()
    df["fingerprint"] = df["canonical_ligand"].map(fingerprint_from_canonical_smiles)
    return df


def summarize_split_df(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "metric": [
                "rows",
                "unique input ligands",
                "unique canonical ligands",
                "duplicate canonical ligands",
                "invalid SMILES rows",
            ],
            "value": [
                len(df),
                df["input_ligand"].nunique(),
                df.loc[df["is_valid_smiles"], "canonical_ligand"].nunique(),
                max(len(df.loc[df["is_valid_smiles"]]) - df.loc[df["is_valid_smiles"], "canonical_ligand"].nunique(), 0),
                int((~df["is_valid_smiles"]).sum()),
            ],
        }
    )


def split_counts(df: pd.DataFrame) -> pd.DataFrame:
    return df[SPLIT_COLUMN].value_counts(dropna=False).rename_axis(SPLIT_COLUMN).reset_index(name="count")


def duplicate_ligands(df: pd.DataFrame) -> pd.DataFrame:
    valid_df = df[df["is_valid_smiles"]].copy()
    return valid_df[valid_df["canonical_ligand"].duplicated(keep=False)].sort_values(["canonical_ligand", SPLIT_COLUMN])


def invalid_ligands(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[~df["is_valid_smiles"], ["input_ligand", SPLIT_COLUMN]].reset_index(drop=True)


def _pairwise_similarity_values(fingerprints: list):
    similarities = []
    for i, fp in enumerate(fingerprints[:-1]):
        similarities.extend(DataStructs.BulkTanimotoSimilarity(fp, fingerprints[i + 1:]))
    return similarities


def _nearest_neighbor_similarities(fingerprints: list):
    if len(fingerprints) < 2:
        return []

    nearest = []
    for i, fp in enumerate(fingerprints):
        other_fps = fingerprints[:i] + fingerprints[i + 1:]
        sims = DataStructs.BulkTanimotoSimilarity(fp, other_fps)
        nearest.append(max(sims) if sims else 0.0)
    return nearest


def _series_stat(values: list[float], stat: str):
    if not values:
        return pd.NA
    series = pd.Series(values, dtype=float)
    if stat == "mean":
        return series.mean()
    if stat == "median":
        return series.median()
    if stat == "p95":
        return series.quantile(0.95)
    if stat == "max":
        return series.max()
    raise ValueError(f"Unsupported stat: {stat}")


def summarize_split_diversity(
    df: pd.DataFrame,
    split_names: list[str] = DIVERSITY_SPLITS,
    similarity_threshold: float = DIVERSITY_THRESHOLD,
    max_ligands_per_split: int = MAX_LIGANDS_PER_SPLIT_FOR_DIVERSITY,
    random_seed: int = DIVERSITY_RANDOM_SEED,
) -> pd.DataFrame:
    valid_df = df[df["is_valid_smiles"]].copy()
    rows = []

    for split_name in split_names:
        split_df = (
            valid_df[valid_df[SPLIT_COLUMN] == split_name]
            .drop_duplicates(subset=["canonical_ligand"])
            .reset_index(drop=True)
        )
        total_unique = len(split_df)
        sampled = total_unique > max_ligands_per_split
        analyzed_df = (
            split_df.sample(n=max_ligands_per_split, random_state=random_seed).reset_index(drop=True)
            if sampled
            else split_df
        )
        fingerprints = analyzed_df["fingerprint"].tolist()
        pairwise = _pairwise_similarity_values(fingerprints)
        nearest = _nearest_neighbor_similarities(fingerprints)

        rows.append(
            {
                "split": split_name,
                "unique_ligands_total": total_unique,
                "ligands_analyzed": len(analyzed_df),
                "sampled": sampled,
                "pair_count": len(pairwise),
                "mean_pairwise_tanimoto": _series_stat(pairwise, "mean"),
                "median_pairwise_tanimoto": _series_stat(pairwise, "median"),
                "p95_pairwise_tanimoto": _series_stat(pairwise, "p95"),
                "max_pairwise_tanimoto": _series_stat(pairwise, "max"),
                "fraction_pairs_ge_threshold": (
                    pd.Series(pairwise, dtype=float).ge(similarity_threshold).mean() if pairwise else pd.NA
                ),
                "mean_nearest_neighbor_tanimoto": _series_stat(nearest, "mean"),
                "median_nearest_neighbor_tanimoto": _series_stat(nearest, "median"),
                "p95_nearest_neighbor_tanimoto": _series_stat(nearest, "p95"),
                "max_nearest_neighbor_tanimoto": _series_stat(nearest, "max"),
            }
        )

    return pd.DataFrame(rows)


def top_similar_pairs_by_split(
    df: pd.DataFrame,
    split_names: list[str] = DIVERSITY_SPLITS,
    top_n: int = TOP_PAIRS_PER_SPLIT,
    max_ligands_per_split: int = MAX_LIGANDS_PER_SPLIT_FOR_DIVERSITY,
    random_seed: int = DIVERSITY_RANDOM_SEED,
) -> pd.DataFrame:
    valid_df = df[df["is_valid_smiles"]].copy()
    rows = []

    for split_name in split_names:
        split_df = (
            valid_df[valid_df[SPLIT_COLUMN] == split_name]
            .drop_duplicates(subset=["canonical_ligand"])
            .reset_index(drop=True)
        )
        if len(split_df) > max_ligands_per_split:
            split_df = split_df.sample(n=max_ligands_per_split, random_state=random_seed).reset_index(drop=True)

        for i in range(len(split_df)):
            left_fp = split_df.loc[i, "fingerprint"]
            for j in range(i + 1, len(split_df)):
                right_fp = split_df.loc[j, "fingerprint"]
                rows.append(
                    {
                        "split": split_name,
                        "left_ligand": split_df.loc[i, "canonical_ligand"],
                        "right_ligand": split_df.loc[j, "canonical_ligand"],
                        "tanimoto": DataStructs.TanimotoSimilarity(left_fp, right_fp),
                    }
                )

    if not rows:
        return pd.DataFrame(columns=["split", "left_ligand", "right_ligand", "tanimoto"])

    pair_df = pd.DataFrame(rows)
    return (
        pair_df.sort_values(["split", "tanimoto", "left_ligand", "right_ligand"], ascending=[True, False, True, True])
        .groupby("split", group_keys=False)
        .head(top_n)
        .reset_index(drop=True)
    )


def find_similar_ligands(df: pd.DataFrame, query_ligands: list[str], top_k: int = 10) -> pd.DataFrame:
    if not query_ligands:
        raise ValueError("Add at least one SMILES string to QUERY_LIGANDS.")

    valid_df = df[df["is_valid_smiles"]].copy()
    if valid_df.empty:
        raise ValueError("The split CSV contains no valid ligand SMILES.")

    rows = []
    for raw_query in query_ligands:
        canonical_query = try_canonicalize_smiles(raw_query)
        if canonical_query is None:
            raise ValueError(f"Could not parse query SMILES: {raw_query}")

        query_fp = fingerprint_from_canonical_smiles(canonical_query)
        scored = valid_df[["input_ligand", "canonical_ligand", SPLIT_COLUMN]].copy()
        scored["query_ligand"] = raw_query
        scored["canonical_query"] = canonical_query
        scored["tanimoto"] = valid_df["fingerprint"].map(lambda fp: DataStructs.TanimotoSimilarity(query_fp, fp))
        scored["exact_canonical_match"] = scored["canonical_ligand"].eq(canonical_query)
        scored = scored.sort_values(
            ["tanimoto", "exact_canonical_match", SPLIT_COLUMN, "canonical_ligand"],
            ascending=[False, False, True, True],
        ).head(top_k)
        rows.append(scored)

    return pd.concat(rows, ignore_index=True)


def pairwise_query_similarity(query_ligands: list[str]) -> pd.DataFrame:
    if not query_ligands:
        return pd.DataFrame()

    canonical_queries = []
    for ligand in query_ligands:
        canonical = try_canonicalize_smiles(ligand)
        if canonical is None:
            raise ValueError(f"Could not parse query SMILES: {ligand}")
        canonical_queries.append(canonical)

    fps = {smiles: fingerprint_from_canonical_smiles(smiles) for smiles in canonical_queries}
    matrix = []
    for left in canonical_queries:
        row = {}
        for right in canonical_queries:
            row[right] = DataStructs.TanimotoSimilarity(fps[left], fps[right])
        matrix.append(pd.Series(row, name=left))

    return pd.DataFrame(matrix)

In [4]:
split_df = load_split_df(SPLIT_CSV_PATH)

display(summarize_split_df(split_df))
display(split_counts(split_df))

print("First few rows:")
display(split_df[["input_ligand", "canonical_ligand", SPLIT_COLUMN, "is_valid_smiles"]].head())

dupes = duplicate_ligands(split_df)
invalid = invalid_ligands(split_df)

print(f"Duplicate canonical ligands: {len(dupes)} rows")
display(dupes.head(20))

print(f"Invalid SMILES rows: {len(invalid)}")
display(invalid.head(20))

,metric,value
0,rows,25902
1,unique input ligands,25902
2,unique canonical ligands,25902
3,duplicate canonical ligands,0
4,invalid SMILES rows,0


,split,count
0,train,17327
1,dropped,8057
2,test,259
3,val,259


First few rows:


,input_ligand,canonical_ligand,split,is_valid_smiles
0,Br.C=CCN1CCc2c(cc(O)c(O)c2Br)C(c2ccccc2)C1,Br.C=CCN1CCc2c(cc(O)c(O)c2Br)C(c2ccccc2)C1,dropped,True
1,Br.C=CCN1CCc2c(cc(O)c(O)c2Cl)C(c2ccccc2)C1,Br.C=CCN1CCc2c(cc(O)c(O)c2Cl)C(c2ccccc2)C1,dropped,True
2,Br.CCCN(CCC)C1CCc2ccc(O)cc2C1,Br.CCCN(CCC)C1CCc2ccc(O)cc2C1,dropped,True
3,Br.CN1CCc2c(cc(O)c(O)c2Br)C(c2ccccc2)C1,Br.CN1CCc2c(cc(O)c(O)c2Br)C(c2ccccc2)C1,dropped,True
4,Br.CN1CCc2c(cc(O)c(O)c2Cl)C(c2ccccc2)C1,Br.CN1CCc2c(cc(O)c(O)c2Cl)C(c2ccccc2)C1,dropped,True


Duplicate canonical ligands: 0 rows


,ligand,split,input_ligand,canonical_ligand,is_valid_smiles,fingerprint


Invalid SMILES rows: 0


,input_ligand,split


In [5]:
print(
    "Within-split diversity summary "
    f"(threshold={DIVERSITY_THRESHOLD}, max ligands analyzed per split={MAX_LIGANDS_PER_SPLIT_FOR_DIVERSITY}):"
)

diversity_summary = summarize_split_diversity(
    split_df,
    split_names=DIVERSITY_SPLITS,
    similarity_threshold=DIVERSITY_THRESHOLD,
    max_ligands_per_split=MAX_LIGANDS_PER_SPLIT_FOR_DIVERSITY,
    random_seed=DIVERSITY_RANDOM_SEED,
)
display(diversity_summary)

print("Most similar ligand pairs found within each split subset:")
display(
    top_similar_pairs_by_split(
        split_df,
        split_names=DIVERSITY_SPLITS,
        top_n=TOP_PAIRS_PER_SPLIT,
        max_ligands_per_split=MAX_LIGANDS_PER_SPLIT_FOR_DIVERSITY,
        random_seed=DIVERSITY_RANDOM_SEED,
    )
)

print(
    "If `sampled` is True, the diversity numbers come from a random subset of that split, "
    "which is usually enough for a sanity check on large splits."
)

Within-split diversity summary (threshold=0.3, max ligands analyzed per split=500):


,split,unique_ligands_total,ligands_analyzed,sampled,pair_count,mean_pairwise_tanimoto,median_pairwise_tanimoto,p95_pairwise_tanimoto,max_pairwise_tanimoto,fraction_pairs_ge_threshold,mean_nearest_neighbor_tanimoto,median_nearest_neighbor_tanimoto,p95_nearest_neighbor_tanimoto,max_nearest_neighbor_tanimoto
0,train,17327,500,True,124750,0.220408,0.213483,0.319444,1.0,0.078076,0.580355,0.583333,0.821468,1.0
1,val,259,259,False,33411,0.153668,0.144737,0.235294,1.0,0.022567,0.586271,0.634146,1.000000,1.0
2,test,259,259,False,33411,0.124460,0.116279,0.209677,1.0,0.012840,0.571601,0.562500,1.000000,1.0


Most similar ligand pairs found within each split subset:


,split,left_ligand,right_ligand,tanimoto
0,test,CC(C)c1ccc(CS(=O)(=O)c2ccc([C@]34CNC[C@H]3C4)cc2)cc1,CC(C)c1ccc(S(=O)(=O)Cc2ccc([C@]34CNC[C@H]3C4)cc2)cc1,1.000000
1,test,CC(N)Cc1c2c(c(Br)c3c1OCC3)OCC2,C[C@@H](N)Cc1c2c(c(Br)c3c1OCC3)OCC2,1.000000
2,test,CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1,CC1=C(/C=C/C(C)=C/C=C/C(C)=C\C(=O)O)C(C)(C)CCC1,1.000000
3,test,C[C@@H]1O[C@H]([C@@H]2CCCN2C)C[S@+]1[O-],C[C@H]1O[C@@H]([C@@H]2CCCN2C)C[S@+]1[O-],1.000000
4,test,C[C@@H]1O[C@H]([C@@H]2CCC[N+]2(C)C)CS1,C[C@@H]1O[C@H]([C@H]2CCC[N+]2(C)C)CS1,1.000000
5,train,COc1ccc(-c2ccccc2N2CCN(CCCCCC(=O)N[C@@H](C)c3ccc(C#N)cc3)CC2)cc1,COc1ccc(-c2ccccc2N2CCN(CCCCCC(=O)N[C@H](C)c3ccc(C#N)cc3)CC2)cc1,1.000000
6,train,Fc1ccc(OC[C@@H]2CC[C@H]3CN(c4ncc(F)cn4)CCN3C2)cc1,Fc1ccc(OC[C@H]2CC[C@@H]3CN(c4ncc(F)cn4)CCN3C2)cc1,1.000000
7,train,O=c1[nH]c2ccc(OCCCN3CCN(c4cccc(Cl)c4Cl)CC3)cc2[nH]1,O=c1[nH]c2ccc(OCCCN3CCCN(c4cccc(Cl)c4Cl)CC3)cc2[nH]1,0.960784
8,train,O=C(CCCCCCCCC(=O)Oc1ccc2c(c1)[C@@]13CCCC[C@H]1[C@@H](C2)N(CC1CC1)CC3)Oc1ccc2c(c1)[C@@]13CCCC[C@H]1[C@@H](C2)N(CC1CC1...,O=C(CCCCCCCC(=O)Oc1ccc2c(c1)[C@@]13CCCC[C@H]1[C@@H](C2)N(CC1CCC1)CC3)Oc1ccc2c(c1)[C@@]13CCCC[C@H]1[C@@H](C2)N(CC1CCC...,0.947368
9,train,O=C(NCC[C@@H](O)CN1CCN(c2cccc(Cl)c2Cl)CC1)c1cc2ccccc2[nH]1,O=C(NCCC(O)CN1CCN(c2cccc(Cl)c2Cl)CC1)c1cc2cc(F)ccc2[nH]1,0.907407


If `sampled` is True, the diversity numbers come from a random subset of that split, which is usually enough for a sanity check on large splits.


In [6]:
if QUERY_LIGANDS:
    print("Pairwise similarity among your query ligands:")
    display(pairwise_query_similarity(QUERY_LIGANDS))

    print(f"Top {TOP_K} closest ligands from the split CSV for each query:")
    display(find_similar_ligands(split_df, QUERY_LIGANDS, top_k=TOP_K))
else:
    print("Add one or more SMILES strings to QUERY_LIGANDS in the config cell, then rerun this cell.")

Pairwise similarity among your query ligands:


,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,C/N=C1/N/C(=C/c2c[nH]c3cc(Br)ccc23)C(=O)N1C,C1=C(c2ccoc2)C2CCN1CC2
Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,1.000000,0.164179,0.185185
C/N=C1/N/C(=C/c2c[nH]c3cc(Br)ccc23)C(=O)N1C,0.164179,1.000000,0.060606
C1=C(c2ccoc2)C2CCN1CC2,0.185185,0.060606,1.000000


Top 10 closest ligands from the split CSV for each query:


,input_ligand,canonical_ligand,split,query_ligand,canonical_query,tanimoto,exact_canonical_match
0,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,1.000000,True
1,Br.O=C1c2ccccc2CCC1C1CCN(CCc2ccccc2)CC1,Br.O=C1c2ccccc2CCC1C1CCN(CCc2ccccc2)CC1,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.652174,False
2,Br.O=C1c2ccccc2CCC1C1CCN(CC2CC2)CC1,Br.O=C1c2ccccc2CCC1C1CCN(CC2CC2)CC1,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.636364,False
3,Cl.O=C1c2ccccc2CCC1C1CCN(Cc2ccccc2)CC1,Cl.O=C1c2ccccc2CCC1C1CCN(Cc2ccccc2)CC1,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.608696,False
4,Cc1ccc(CN2CCC(C3CCc4ccccc4C3=O)CC2)cc1.O,Cc1ccc(CN2CCC(C3CCc4ccccc4C3=O)CC2)cc1.O,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.595745,False
5,CC(C)(C)c1ccc(CN2CCC(C3CCc4ccccc4C3=O)CC2)cc1,CC(C)(C)c1ccc(CN2CCC(C3CCc4ccccc4C3=O)CC2)cc1,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.560000,False
6,Cl.O=C1c2ccccc2CCC1C1CCN(Cc2ccc(F)cc2)CC1,Cl.O=C1c2ccccc2CCC1C1CCN(Cc2ccc(F)cc2)CC1,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.529412,False
7,O=C1c2ccc(F)cc2CCC1C1CCN(CCc2ccccc2)CC1,O=C1c2ccc(F)cc2CCC1C1CCN(CCc2ccccc2)CC1,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.529412,False
8,O=C1c2ccc(F)cc2CCC1C1CCN(Cc2ccccc2)CC1,O=C1c2ccc(F)cc2CCC1C1CCN(Cc2ccccc2)CC1,train,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.509804,False
9,COc1ccc2c(c1)CCC(C1CCN(CCc3ccccc3)CC1)C2=O,COc1ccc2c(c1)CCC(C1CCN(CCc3ccccc3)CC1)C2=O,dropped,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,Br.CC(C1CC1)N1CCC(C2CCc3ccccc3C2=O)CC1,0.509091,False


In [7]:
# split_df[split_df['split'] == 'train'].head()
# split_df[split_df['split'] == 'val'].head()
split_df[split_df['split'] == 'test'].head()


,ligand,split,input_ligand,canonical_ligand,is_valid_smiles,fingerprint
8057,Brc1ccc(-[n+]2cc[n+](Cc3ccccc3)cc2)c2cc[nH]c12,test,Brc1ccc(-[n+]2cc[n+](Cc3ccccc3)cc2)c2cc[nH]c12,Brc1ccc(-[n+]2cc[n+](Cc3ccccc3)cc2)c2cc[nH]c12,True,"[0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0..."
8058,C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=Cc5oncc5C[C@]4(C)[C@H]3CC[C@@]21C,test,C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=Cc5oncc5C[C@]4(C)[C@H]3CC[C@@]21C,C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=Cc5oncc5C[C@]4(C)[C@H]3CC[C@@]21C,True,"[0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
8059,C1=C(c2c[nH]c3cnccc23)CNCC1,test,C1=C(c2c[nH]c3cnccc23)CNCC1,C1=C(c2c[nH]c3cnccc23)CNCC1,True,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
8060,C1=C(c2ccoc2)C2CCN1CC2,test,C1=C(c2ccoc2)C2CCN1CC2,C1=C(c2ccoc2)C2CCN1CC2,True,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0..."
8061,C1C2C3CC4C2C2C1C3[C@H](NCC13C5C6C7C5C1C7C63)C42,test,C1C2C3CC4C2C2C1C3[C@H](NCC13C5C6C7C5C1C7C63)C42,C1C2C3CC4C2C2C1C3[C@H](NCC13C5C6C7C5C1C7C63)C42,True,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
